In [3]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

# Keys you still need
for key in ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]:
    print(key, "OK" if os.getenv(key) else "MISSING")

TAVILY_API_KEY OK
LANGSMITH_API_KEY MISSING


In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",            # use the exact name from `ollama list`
    temperature=0,
    reasoning=False,             # turn off Qwen3's thinking mode
    validate_model_on_init=True, # error right away if the model name is wrong
)

print(llm.invoke("Reply with just the word OK").content)

OK


In [6]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="rag-chroma",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)
retriever = vectorstore.as_retriever()
print("Loaded:", vectorstore._collection.count(), "chunks")   # expect 547

Loaded: 547 chunks


In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

urls = [
    # Anthropic engineering: agents, context, tools, MCP
    "https://www.anthropic.com/engineering/building-effective-agents",                  # Dec 2024
    "https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents", # Sep 2025
    "https://www.anthropic.com/engineering/multi-agent-research-system",                # Jun 2025
    "https://www.anthropic.com/engineering/code-execution-with-mcp",                    # Nov 2025
    "https://www.anthropic.com/engineering/writing-tools-for-agents",                   # 2025 (unverified)
    "https://www.anthropic.com/news/contextual-retrieval",                              # Sep 2024, RAG technique
    # Lilian Weng: reasoning, reward hacking, hallucination
    "https://lilianweng.github.io/posts/2025-05-01-thinking/",                          # May 2025
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",                    # Nov 2024
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",                     # Jul 2024
    # Evals
    "https://hamel.dev/blog/posts/evals-faq/",                                          # expanded Sep 2026
    "https://hamel.dev/blog/posts/evals/",                                              # 2024 classic
    # Agents
    "https://huyenchip.com/2025/01/07/agents.html",                                     # Jan 2025 (unverified)
]

# Load each URL separately so one bad URL doesn't stop the whole cell
docs_list = []
for url in urls:
    try:
        loaded = WebBaseLoader(url).load()
        chars = sum(len(d.page_content) for d in loaded)
        print(f"{chars:>8} chars  {url}")
        if chars < 2000:
            print("          ^ suspiciously short, skipping (blocked page or wrong URL?)")
            continue
        docs_list.extend(loaded)
    except Exception as e:
        print(f"  FAILED  {url} -> {type(e).__name__}: {e}")

print(f"\nLoaded {len(docs_list)} documents")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)
print(f"Split into {len(doc_splits)} chunks")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="rag-chroma",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)
vectorstore.reset_collection()

batch_size = 64
for i in range(0, len(doc_splits), batch_size):
    vectorstore.add_documents(doc_splits[i : i + batch_size])
    print(f"Indexed {min(i + batch_size, len(doc_splits))}/{len(doc_splits)}")

retriever = vectorstore.as_retriever()
print("Total in store:", vectorstore._collection.count())

USER_AGENT environment variable not set, consider setting it to identify your requests.


KeyboardInterrupt: 

In [ ]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
print(len(embeddings.embed_query("hello world")))   # expect 768

768


In [ ]:
lengths = sorted((len(d.page_content), i) for i, d in enumerate(doc_splits))
print("chunks:", len(doc_splits))
print("longest 5 (chars, index):", lengths[-5:])

chunks: 547
longest 5 (chars, index): [(1429, 10), (1455, 48), (1480, 49), (1483, 18), (1613, 50)]


In [7]:
import os
from typing import Literal

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

LOCAL_LLM = os.getenv("LOCAL_LLM", "qwen3:8b")  # add LOCAL_LLM=qwen3:8b to .env


class GradeDocument(BaseModel):
    """Relevance check for a retrieved document."""
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document is relevant to the question, otherwise 'no'"
    )


llm = ChatOllama(model=LOCAL_LLM, temperature=0, reasoning=False)
structured_grader_llm = llm.with_structured_output(GradeDocument, method="json_schema")

grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader assessing whether a retrieved document is relevant to a user question. "
     "If the document contains information or keywords related to the question, grade it as relevant. "
     "It does not need to be a strict test; the goal is to filter out clearly wrong retrievals."),
    ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}"),
])

retrieval_grader = grade_prompt | structured_grader_llm

In [ ]:
question = "How do AI agents manage memory and context?"
docs = retriever.invoke(question)

for d in docs:
    result = retrieval_grader.invoke({"question": question, "document": d.page_content})
    print(result.binary_score, "|", d.metadata["source"], "|", d.page_content[:80].replace("\n", " "))

# Negative test: an unrelated question should get "no"


yes | https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents | components of context (system prompts, tools, examples, message history, etc) is
yes | https://huyenchip.com/2025/01/07/agents.html | This post covered conceptually how agents work and different components of an ag
yes | https://www.anthropic.com/news/contextual-retrieval | Contextual retrieval in AI systems \ AnthropicSkip to main contentNewsResearchPo
yes | https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents | history, etc).An agent running in a loop generates more and more data that could


In [ ]:
off_topic = retrieval_grader.invoke({"question": "What is a good biryani recipe?",
                                     "document": docs[0].page_content})
print("is the question relevant?", off_topic.binary_score)

off-topic -> no


In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

REFUSAL = (
    "Sorry, I couldn't find information about that in my knowledge base, "
    "so I'm not able to answer it reliably. Feel free to ask about AI agents, "
    "context engineering, LLM reasoning, or evaluation."
)

generate_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the documents provided by the user.\n\n"
     "Rules:\n"
     "1. Every statement in your answer must be supported by the documents. "
     "Do not use outside or prior knowledge, even if you are confident you know the answer.\n"
     "2. The documents come from an automatic search and may be irrelevant to the question. "
     "First decide whether they actually contain the answer.\n"
     f"3. If they do not, reply with exactly: \"{REFUSAL}\" and nothing else.\n"
     "4. If they do, answer in at most three sentences and cite the supporting "
     "document numbers, like [1] or [2][3]."),
    ("human", "<documents>\n{context}\n</documents>\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(
        f'<document index="{i}" source="{d.metadata.get("source", "unknown")}">\n'
        f"{d.page_content}\n</document>"
        for i, d in enumerate(docs, start=1)
    )


rag_chain = generate_prompt | llm | StrOutputParser()

In [13]:
question = "How do AI agents manage memory and context?"
docs = retriever.invoke(question)
generation = rag_chain.invoke({"context": format_docs(docs), "question": question})
print(generation)

AI agents manage memory and context by dynamically retrieving information at runtime and maintaining a "just in time" approach to context, which involves supplementing the model's context with a memory system to handle information that exceeds the model's context limit [2]. Context engineering is also crucial, as it involves curating the limited context window from an evolving universe of information, treating context as a finite resource with diminishing marginal returns [4].


In [14]:
question = "Who won the 2022 FIFA World Cup?"
docs = retriever.invoke(question)
print(rag_chain.invoke({"context": format_docs(docs), "question": question}))

Sorry, I couldn't find information about that in my knowledge base, so I'm not able to answer it reliably. Feel free to ask about AI agents, context engineering, LLM reasoning, or evaluation.


In [11]:
for d in docs:
    print(retrieval_grader.invoke({"question": question, "document": d.page_content}).binary_score)

no
no
no
no


In [15]:
class GradeHallucination(BaseModel):
    """Whether the answer is supported by the documents."""
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if every factual claim in the answer is supported by the documents, otherwise 'no'"
    )


class GradeAnswer(BaseModel):
    """Whether the answer resolves the question."""
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the answer resolves the question, otherwise 'no'"
    )


hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict grader checking whether an answer is grounded in a set of documents. "
     "Check each factual claim in the answer. Answer 'yes' only if every claim is supported by the documents. "
     "If any claim relies on information not found in the documents, answer 'no', "
     "even if the claim is true in the real world."),
    ("human", "<documents>\n{documents}\n</documents>\n\nAnswer to check:\n{generation}"),
])

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader checking whether an answer resolves a user's question. "
     "Answer 'yes' if it directly addresses and answers the question. "
     "Answer 'no' if it is off-topic, only partially relevant, or declines to answer."),
    ("human", "Question: {question}\n\nAnswer:\n{generation}"),
])

hallucination_grader = hallucination_prompt | llm.with_structured_output(GradeHallucination, method="json_schema")
answer_grader = answer_prompt | llm.with_structured_output(GradeAnswer, method="json_schema")

In [16]:
agent_q = "How do AI agents manage memory and context?"
agent_docs = format_docs(retriever.invoke(agent_q))
good_answer = rag_chain.invoke({"context": agent_docs, "question": agent_q})

# 1. Real grounded answer -> hallucination: yes, answer: yes
print("grounded:", hallucination_grader.invoke({"documents": agent_docs, "generation": good_answer}).binary_score)
print("useful:  ", answer_grader.invoke({"question": agent_q, "generation": good_answer}).binary_score)

# 2. True-but-ungrounded answer vs agent docs -> hallucination: no
fake = "Argentina won the 2022 FIFA World Cup, beating France on penalties."
print("argentina grounded:", hallucination_grader.invoke({"documents": agent_docs, "generation": fake}).binary_score)

# 3. Refusal -> answer: no
print("refusal useful:", answer_grader.invoke({"question": agent_q, "generation": REFUSAL}).binary_score)

grounded: yes
useful:   yes
argentina grounded: no
refusal useful: no


In [17]:
class RouteQuery(BaseModel):
    """Where to send the user's question."""
    datasource: Literal["vectorstore", "web_search"] = Field(
        description="'vectorstore' for questions about the knowledge base topics, otherwise 'web_search'"
    )


router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You route a user question to either a vectorstore or web search.\n\n"
     "The vectorstore contains articles on: AI agents and agent design patterns, "
     "context engineering, multi-agent systems, tool design for agents and MCP, "
     "RAG and contextual retrieval, LLM reasoning and test-time compute, "
     "reward hacking, hallucinations in LLMs, and LLM evaluation (evals).\n\n"
     "Rules:\n"
     "1. Use 'vectorstore' for questions related to these topics. "
     "Be lenient: the question does not need to use the exact keywords.\n"
     "2. Use 'web_search' for anything else, and for questions about recent news, "
     "current events, or the latest releases, even if the topic matches, "
     "because the vectorstore is a fixed snapshot."),
    ("human", "{question}"),
])

question_router = router_prompt | llm.with_structured_output(RouteQuery, method="json_schema")

In [18]:
tests = {
    "How do AI agents manage memory and context?": "vectorstore",
    "What is LLM-as-a-judge?":                      "vectorstore",
    "Why do reasoning models think before answering?": "vectorstore",
    "Who won the 2022 FIFA World Cup?":             "web_search",
    "What's the weather in Karachi today?":         "web_search",
    "What AI models were released this week?":      "web_search",   # topic matches, but recency
}

for q, expected in tests.items():
    got = question_router.invoke({"question": q}).datasource
    print("✅" if got == expected else "❌", f"{got:<12} | {q}")

✅ vectorstore  | How do AI agents manage memory and context?
✅ vectorstore  | What is LLM-as-a-judge?
✅ vectorstore  | Why do reasoning models think before answering?
✅ web_search   | Who won the 2022 FIFA World Cup?
✅ web_search   | What's the weather in Karachi today?
✅ web_search   | What AI models were released this week?


In [19]:
from langchain_core.documents import Document
from langchain_tavily import TavilySearch

web_search_tool = TavilySearch(max_results=3, topic="general")


def web_search(query: str) -> list[Document]:
    """Search the web and return results as Documents, same shape as the retriever."""
    response = web_search_tool.invoke({"query": query})
    return [
        Document(
            page_content=r["content"],
            metadata={"source": r["url"], "title": r.get("title", "")},
        )
        for r in response["results"]
    ]

In [20]:
question = "Who won the 2022 FIFA World Cup?"
web_docs = web_search(question)

for d in web_docs:
    print(d.metadata["source"], "|", d.page_content[:100].replace("\n", " "))

print("\n->", rag_chain.invoke({"context": format_docs(web_docs), "question": question}))

https://en.wikipedia.org/wiki/2022_FIFA_World_Cup | The final was played on 18 December between Argentina and France. Both teams had won the event twice
https://en.wikipedia.org/wiki/2022_FIFA_World_Cup_final | | Gonzalo Montiel's match-winning penalty kick in the penalty shoot-out | | Event | 2022 FIFA World 
https://www.youtube.com/watch?v=EROb-E23ZVs | # The Moment When Argentina Won The 2022 FIFA World Cup ## Channel: FIFA (verified) 35.8M subscriber

-> Argentina won the 2022 FIFA World Cup [1][2].
